# React 框架 · 第 11 天

## React 与 TypeScript：Hooks、Props 和事件类型

**资料依据**：`React+TS.pdf`（15 页）、当天 3 篇 MD 笔记、`react-ts-pro` 的 TSX 示例及配置。课件项目使用 React 18.2、TypeScript 5.0、Vite 4.4，并开启 `strict` 类型检查。

**学习主线**：类型从数据入口和状态开始，沿着 props、回调与事件流动；对可能为空的 ref 和异步数据做好检查。TypeScript 在开发和构建时发现类型问题，不能替代运行时的数据校验。

## 一、本日知识地图

![课程资料中的 React 与 TypeScript 知识地图](assets/day11/1.jpeg)

重点分成两条线：

| 位置 | 要解决的问题 | 常用写法 |
|---|---|---|
| `useState` | 状态及 setter 可接收什么值 | 自动推断、泛型、联合类型 |
| `useRef` | `.current` 指向哪类 DOM 或保存何种值 | DOM 元素泛型、可空类型 |
| 组件 Props | 父组件能传什么、哪些属性必填 | `type` / `interface` |
| `children` | 组件标签中能放什么内容 | `React.ReactNode` |
| 事件与回调 Props | 事件对象、子传父的数据是什么类型 | React 事件类型、函数签名 |

类型标注优先写在**边界**：状态初始值无法准确推断处、组件对外接口、独立事件处理函数。其余让 TS 推断，避免无意义的重复标注。

## 二、用 Vite 创建课程项目

课件使用 React + TypeScript 模板：

```bash
npm create vite@latest react-ts-pro -- --template react-ts
cd react-ts-pro
npm install
npm run dev
```

`react-ts-pro` 是项目目录名，可自行替换；`react-ts` 指定模板。课程项目主要文件：

```text
src/main.tsx    渲染入口
src/App.tsx     根组件
tsconfig.json   TS 编译与检查配置
package.json    开发、构建等脚本
```

课件 `package.json` 的构建脚本是 `tsc && vite build`：先检查类型，再打包。`strict: true` 会使空值、参数等问题更早暴露。运行 `npm run dev` 便于练习，完成后用 `npm run build` 检查类型与构建；模板命令的实际交互以本机运行结果为准。

## 三、`useState`：初始值明确时交给类型推断

```tsx
import { useState } from 'react'

function Demo() {
  const [visible, setVisible] = useState(false) // boolean
  const [scores, setScores] = useState([1, 2, 3]) // number[]

  return (
    <>
      <button onClick={() => setVisible(v => !v)}>
        {visible ? '隐藏' : '显示'}
      </button>
      <button onClick={() => setScores(list => [...list, 4])}>
        {scores.join(', ')}
      </button>
    </>
  )
}
```

TS 根据初始值推断状态类型，`setVisible('yes')`、`setScores(['x'])` 会报类型错误。setter 既能接收新值，也能接收从旧值计算新值的函数。后一种写法适合计数累加、追加列表等依赖先前状态的更新。

若以空数组开始且没有足够类型信息，推断可能过窄；应显式写成 `useState<Item[]>([])`。

## 四、`useState<T>`：对象状态的明确契约

```tsx
type User = {
  id: number
  name: string
  age: number
}

function Profile() {
  const [user, setUser] = useState<User>({
    id: 1,
    name: 'Jack',
    age: 18,
  })

  const growUp = () => {
    setUser(current => ({ ...current, age: current.age + 1 }))
  }

  return <button onClick={growUp}>{user.name}：{user.age}</button>
}
```

泛型 `User` 同时约束初始状态和之后的更新。对象更新须返回新对象，不能直接改 `user.age`。`type` 和 `interface` 都可定义对象形状；此处与课程一样从简单对象类型开始。

**小提醒**：资料中有 `String`、`Number` 大写写法。描述普通字符串和数字应使用 `string`、`number` 原始类型。

## 五、初始值为空：用联合类型表达真实状态

用户数据在请求返回前不存在，状态就应允许 `null`：

```tsx
type User = { name: string; age: number }

function Profile() {
  const [user, setUser] = useState<User | null>(null)

  // 请求成功后：setUser({ name: 'Jack', age: 18 })
  // 退出登录后：setUser(null)
  return <p>{user ? `${user.name} · ${user.age}` : '尚无用户'}</p>
}
```

`user.name` 不能在未判断时直接访问，因为 `user` 可能为 `null`。可按需要用 `if (user) { ... }`、条件渲染或 `user?.name`；但可选链得到的可能是 `undefined`，界面通常仍需提供空状态。不要用非空断言 `user!.name` 掩盖真实的空值可能。

若空状态用 `undefined` 表示，就写 `User | undefined`；类型应反映实际数据生命周期。

## 六、`useRef` 读取 DOM：注明元素类型并处理空值

```tsx
import { useEffect, useRef } from 'react'

function SearchBox() {
  const inputRef = useRef<HTMLInputElement>(null)

  useEffect(() => {
    inputRef.current?.focus()
  }, [])

  return <input ref={inputRef} placeholder="搜索" />
}
```

`HTMLInputElement` 让 TS 知道 `.current` 可调用 `focus()`，且能给出输入框相关属性提示。初始值为 `null`，渲染前或卸载后也可能没有元素，因此访问前需空值检查。

资料里有 `<div ref={inputRef}>` 却把 ref 标成 `HTMLDivElement`、变量命名为 `inputRef` 的混搭示例；元素类型应与实际 JSX 对应。资料还展示 `null!` 绕过检查；练习中优先用可选链或显式判断，不要依赖断言逃避空值。

## 七、`useRef` 保存稳定值：定时器例子

ref 的另一种用途是保存**不需要触发渲染**的可变值，例如定时器 ID：

```tsx
import { useEffect, useRef } from 'react'

function Clock() {
  const timerRef = useRef<number | null>(null)

  useEffect(() => {
    timerRef.current = window.setInterval(() => {
      console.log('tick')
    }, 1000)

    return () => {
      if (timerRef.current !== null) {
        window.clearInterval(timerRef.current)
      }
    }
  }, [])

  return <p>查看控制台</p>
}
```

浏览器中的 `window.setInterval` 返回数字，因而此处用 `number | null`。如果项目同时使用 Node 的定时器类型，可用 `ReturnType<typeof setInterval>` 配合对应环境。修改 `.current` 不会让组件重渲染；要显示在界面上的值应放在 state 中。

## 八、`type` / `interface` 给组件 Props 建立边界

```tsx
interface ButtonProps {
  className: string
  title?: string
  disabled?: boolean
}

function Button({ className, title, disabled }: ButtonProps) {
  return (
    <button className={className} title={title} disabled={disabled}>
      提交
    </button>
  )
}

// 正确：<Button className="primary" />
// 错误：<Button title="缺少必填 className" />
```

`className` 必填；加 `?` 的属性可省略，读取时其类型包含 `undefined`。使用 `type ButtonProps = { ... }` 也能完成课件中的同类任务。类型检查发生在开发/构建阶段，来自接口或浏览器存储的实际数据仍须按业务需要校验。

## 九、为 `children` 标注 `React.ReactNode`

```tsx
import type { ReactNode } from 'react'

type CardProps = {
  title: string
  children: ReactNode
}

function Card({ title, children }: CardProps) {
  return (
    <section>
      <h2>{title}</h2>
      <div>{children}</div>
    </section>
  )
}

// <Card title="简介"><strong>你好</strong></Card>
// <Card title="数量">3</Card>
```

`ReactNode` 可表示常见的可渲染内容，如元素、字符串、数字、片段、空值等。这里 `children` 是必填；若允许空卡片可写 `children?: ReactNode`。当接口明确要求“恰好一个 React 元素”时，再选择更窄的类型。不要认为 `children` 一定是 DOM 元素或一定拥有 `.props`。

## 十、事件回调 Prop：把子传父的数据写清楚

```tsx
type SenderProps = {
  onSend?: (message: string) => void
}

function Sender({ onSend }: SenderProps) {
  return (
    <button onClick={() => onSend?.('来自子组件的消息')}>
      发送
    </button>
  )
}

function App() {
  const handleSend = (message: string) => console.log(message)
  return <Sender onSend={handleSend} />
}
```

`onSend` 是函数类型，参数是 `string`，返回值不被调用方使用，因此写 `void`。加 `?` 后回调可省略，调用时用 `?.` 或先判断。父组件传入的函数负责处理数据，子组件只按照约定发出消息。内联写 `<Sender onSend={message => console.log(message)} />` 时，`message` 可从 Props 签名推断成 `string`。

## 十一、DOM 事件类型：内联推断与独立处理函数

JSX 中直接写事件处理函数时，TS 通常能从标签推断事件类型；抽出函数时可显式标注：

```tsx
import type { ChangeEvent, MouseEventHandler } from 'react'

function Search() {
  const handleChange = (event: ChangeEvent<HTMLInputElement>) => {
    console.log(event.currentTarget.value)
  }

  const handleClick: MouseEventHandler<HTMLButtonElement> = event => {
    console.log(event.currentTarget.disabled)
  }

  return (
    <>
      <input onChange={handleChange} />
      <button onClick={handleClick}>搜索</button>
    </>
  )
}
```

`ChangeEvent<HTMLInputElement>` 描述输入框变更事件；`MouseEventHandler<HTMLButtonElement>` 描述按钮点击回调。`currentTarget` 是绑定事件处理器的元素，类型更明确；`target` 可能是事件源内部的后代元素，不应总假定它就是按钮或输入框。

## 十二、一个小例子串起状态、Props 与事件

```tsx
import { useState } from 'react'
import type { ChangeEvent, ReactNode } from 'react'

type User = { name: string; age: number }

type PanelProps = {
  title: string
  children: ReactNode
  onSave: (user: User) => void
}

function Panel({ title, children, onSave }: PanelProps) {
  const [name, setName] = useState('')

  const handleChange = (event: ChangeEvent<HTMLInputElement>) => {
    setName(event.currentTarget.value)
  }

  return (
    <section>
      <h2>{title}</h2>
      {children}
      <input value={name} onChange={handleChange} />
      <button onClick={() => onSave({ name, age: 18 })}>保存</button>
    </section>
  )
}
```

数据流是：输入事件 → `setName` 更新状态 → 点击时构造符合 `User` 的对象 → 调用父组件提供的 `onSave`。每个边界都有相应类型；组件内部的 `name` 可由初始值自动推断。

## 十三、看懂课程示例中的几个问题

- **`useState<User>(null)`**：在 `strict` 下 `null` 不能赋给 `User`；应写 `useState<User | null>(null)`。
- **`String` / `Number`**：这两个大写名称表示包装对象相关类型，普通业务字段用 `string` / `number`。
- **`ref.current.focus()`**：`.current` 可能为 `null`，先检查或使用 `?.`。
- **DOM 泛型与 JSX 不符**：`ref<HTMLDivElement>` 应配 `<div>`，输入框配 `HTMLInputElement`。
- **`null!` 非空断言**：只让编译器放行，并不保证运行时真的存在；能判断就判断。
- **`useEffect` 中定时器不清理**：组件卸载后仍运行；应在清理函数里 `clearInterval`。
- **可选回调直接调用**：`onSend?: ...` 需要 `onSend?.(...)` 或条件判断。
- **误把类型当运行时验证**：API 返回值即使被声明为 `User`，仍可能缺字段；不可信外部数据需要实际验证。

## 十四、练习与自检

1. 创建 `react-ts` 模板项目，确认 `npm run dev` 能启动，再执行 `npm run build`。
2. 写一个 `useState<Product[]>([])` 商品列表，尝试把价格字符串放进 `number` 字段，观察报错并修正。
3. 写 `useState<User | null>(null)`，分别展示“未加载”和“已加载”的界面，不使用非空断言。
4. 封装 `SearchInput`：`placeholder` 可选，`onSearch: (keyword: string) => void` 必填；输入框 `ref` 在挂载后聚焦。
5. 写按钮点击和输入变更的独立处理函数，分别为事件标注正确的 React 类型。
6. 建立一个定时器 ref，在卸载时清理；解释为什么 timer ID 不用 `useState`。

自检时能说明：**这个类型从哪里来、何时可能为空、谁负责调用回调、类型错误在什么阶段被发现。**

## 十五、本日总结

React + TypeScript 的关键是让数据契约跟着状态和组件接口走：明确初始值让 TS 推断，复杂对象用泛型，缺省数据用联合类型；DOM ref 写具体元素类型并检查空值，稳定存储用可变 ref；Props 用 `type` 或 `interface` 定义，`children` 常用 `ReactNode`，回调和 DOM 事件写清参数类型。先保证数据流正确、类型表达真实状态，再利用编译检查减少错误。